# NYISO Solar Forecasting: Physics-Informed Feature Engineering for Residual Learning

## Imports and Configuration

In [9]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [10]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_NYISOSolarForecast"

data_root = repo_root / "data"
processed_dir = data_root / "processed"

merged_out = processed_dir / "03_merged_data.csv"

feature_out_csv = processed_dir / "04_system_features.csv"
feature_out_parquet = processed_dir / "04_system_features.parquet"
feature_metadata_out = processed_dir / "04_system_features_meta.json"

In [11]:
TARGET_MODE = "residual"
KEEP_DAYLIGHT_ONLY = False
USE_CAPACITY_FEATURES = False
FORWARD_FILL_CAPACITY = False
ADD_LAG_FEATURES = True
ADD_ROLLING_FEATURES = True
DROP_ROWS_WITH_TARGET_NA = True

LAG_HOURS = [1, 2, 3, 6, 12, 24]
ROLLING_WINDOWS = [3, 6, 12, 24]

TRAIN_END = "2023-12-31 23:00:00"
VALID_END = "2024-12-31 23:00:00"

RANDOM_STATE = 42

In [12]:
df = pd.read_csv(merged_out, low_memory=False)

In [13]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

df["time_stamp"] = pd.to_datetime(df["time_stamp"], utc=True, errors="coerce")

if "time" in df.columns:
    df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
    same_time_mask = (df["time"] == df["time_stamp"]) | (
        df["time"].isna() & df["time_stamp"].isna()
    )
    if bool(same_time_mask.all()):
        df = df.drop(columns=["time"])

numeric_cols = [
    "actual_mw",
    "forecast_mw",
    "capacity_mw",
    "temperature_2m",
    "surface_pressure",
    "cloud_cover",
    "windspeed_10m",
    "shortwave_radiation",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["zone_name"] = df["zone_name"].astype(str).str.strip().str.upper()

In [14]:
df["time_local"] = df["time_stamp"].dt.tz_convert("America/New_York")
df["date_local"] = df["time_local"].dt.date
df["year_from_ts"] = df["time_local"].dt.year
df["quarter_local"] = df["time_local"].dt.quarter
df["month_local"] = df["time_local"].dt.month
df["day_local"] = df["time_local"].dt.day
df["dayofweek_local"] = df["time_local"].dt.dayofweek
df["hour_local"] = df["time_local"].dt.hour
df["is_weekend"] = df["dayofweek_local"].isin([5, 6]).astype(int)
df["is_daylight_proxy"] = (df["shortwave_radiation"] > 0).astype(int)

In [15]:
df["forecast_error_mw"] = df["actual_mw"] - df["forecast_mw"]
df["absolute_error_mw"] = (df["actual_mw"] - df["forecast_mw"]).abs()
df["smape"] = np.where(
    (df["actual_mw"].abs() + df["forecast_mw"].abs()) > 0,
    200 * df["absolute_error_mw"] / (df["actual_mw"].abs() + df["forecast_mw"].abs()),
    np.nan
)

In [16]:
df_system = (
    df[df["zone_name"] == "SYSTEM"]
    .copy()
    .sort_values("time_stamp")
    .reset_index(drop=True)
)

print("System Shape:", df_system.shape)
print("System Time Range:", df_system["time_stamp"].min(), "to", df_system["time_stamp"].max())

System Shape: (42455, 24)
System Time Range: 2020-11-17 05:00:00+00:00 to 2025-09-21 03:00:00+00:00


In [18]:
df_system["dayofyear_local"] = df_system["time_local"].dt.dayofyear
df_system["weekofyear_local"] = df_system["time_local"].dt.isocalendar().week.astype(int)

df_system["hour_sin"] = np.sin(2 * np.pi * df_system["hour_local"] / 24)
df_system["hour_cos"] = np.cos(2 * np.pi * df_system["hour_local"] / 24)

df_system["month_sin"] = np.sin(2 * np.pi * df_system["month_local"] / 12)
df_system["month_cos"] = np.cos(2 * np.pi * df_system["month_local"] / 12)

df_system["dayofyear_sin"] = np.sin(2 * np.pi * df_system["dayofyear_local"] / 365.25)
df_system["dayofyear_cos"] = np.cos(2 * np.pi * df_system["dayofyear_local"] / 365.25)

df_system["dayofweek_sin"] = np.sin(2 * np.pi * df_system["dayofweek_local"] / 7)
df_system["dayofweek_cos"] = np.cos(2 * np.pi * df_system["dayofweek_local"] / 7)

In [20]:
df_system["is_morning_ramp"] = df_system["hour_local"].between(6, 9).astype(int)
df_system["is_midday"] = df_system["hour_local"].between(10, 14).astype(int)
df_system["is_evening_ramp"] = df_system["hour_local"].between(15, 18).astype(int)
df_system["is_night"] = (df_system["is_daylight_proxy"] == 0).astype(int)

In [22]:
if TARGET_MODE == "residual":
    df_system["target"] = df_system["forecast_error_mw"]
elif TARGET_MODE == "actual":
    df_system["target"] = df_system["actual_mw"]
else:
    raise ValueError("TARGET_MODE must be either 'residual' or 'actual'.")

print("Target Mode:", TARGET_MODE)
print(df_system["target"].describe())

Target Mode: residual
count    41455.000000
mean        -2.175051
std        166.121605
min      -1308.850000
25%        -18.105000
50%          0.000000
75%          0.000000
max       1724.350000
Name: target, dtype: float64
